# Colab A100 Unsloth Qwen Finance

Thin notebook for VS Code + Google Colab execution. Reusable logic lives in repo scripts under `scripts/colab/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get('ML_CLASS_LORA_REPO', 'https://github.com/nathanaelguitar/ML_Class_LORA.git')
REPO_BRANCH = os.environ.get('ML_CLASS_LORA_BRANCH', 'main')
REPO_DIR = Path('/content/ML_Class_LORA')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repo_dir', REPO_DIR)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!python scripts/colab/verify_runtime.py --require-gpu-substring A100 --min-memory-gb 35


In [ ]:
!python scripts/colab/bootstrap_unsloth_env.py


In [ ]:
import importlib.metadata
import torch

print('torch', torch.__version__)
print('cuda', torch.version.cuda)
print('bf16', torch.cuda.is_bf16_supported())
print('unsloth', importlib.metadata.version('unsloth'))


In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path('config/colab_paths.example.yaml')
cfg = yaml.safe_load(CONFIG_PATH.read_text())
for key in ('train_file', 'train_eval_file', 'val_file', 'test_file'):
    path = Path(cfg['datasets'][key])
    size_mb = round(path.stat().st_size / (1024 ** 2), 2) if path.exists() else None
    print(key, path, 'exists=', path.exists(), 'size_mb=', size_mb)


In [ ]:
from huggingface_hub import snapshot_download
import yaml

cfg = yaml.safe_load(CONFIG_PATH.read_text())
snapshot_download(repo_id=cfg['model']['model_id'], local_files_only=cfg['model'].get('local_files_only', False))


In [ ]:
import os

if os.environ.get('HF_TOKEN'):
    get_ipython().system('python scripts/colab/check_hf_auth.py --token-env HF_TOKEN')
else:
    print('HF_TOKEN not set; skipping Hugging Face auth check.')


In [ ]:
from pathlib import Path

for key in ('adapters', 'checkpoints', 'outputs', 'manifests'):
    path = Path(cfg['drive'][key])
    print(key, path, 'exists=', path.exists())
    if path.exists():
        children = sorted(item.name for item in path.iterdir())[:10]
        print('  first_entries=', children)


In [ ]:
RUN_NAME = 'qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z'
SANITY_STEPS = 5
get_ipython().system(f'python scripts/colab/sanity_gate.py --config config/colab_paths.example.yaml --run-name {RUN_NAME} --sanity-steps {SANITY_STEPS}')


In [ ]:
import json
from pathlib import Path

manifest_path = Path(cfg['drive']['manifests']) / f"{RUN_NAME}-sanity_manifest.json"
payload = json.loads(manifest_path.read_text())
print('manifest_path', manifest_path)
for row in payload['generations']:
    print('example', row['example_index'])
    print('generated_output', row['generated_output'][:800])
    print('---')


In [ ]:
# Launch this only after the sanity gate succeeds.
# RUN_NAME matches the existing local 71k-step (checkpoint-4500) WRDS run;
# --resume-latest picks up training from that checkpoint instead of starting over.
get_ipython().system(f'python scripts/colab/train_unsloth_from_config.py --config config/colab_paths.example.yaml --run-name {RUN_NAME} --resume-latest')


In [ ]:
get_ipython().system(f'python scripts/colab/run_post_training_evals.py --config config/colab_paths.example.yaml --run-name {RUN_NAME}')


In [ ]:
import os
import subprocess

ENABLE_HF_UPLOAD = os.environ.get('ENABLE_HF_UPLOAD', '').lower() in {'1', 'true', 'yes'}
if ENABLE_HF_UPLOAD:
    subprocess.run([
        'python',
        'scripts/colab/push_final_adapter.py',
        '--config',
        'config/colab_paths.example.yaml',
        '--run-name',
        RUN_NAME,
        '--enable-upload',
    ], check=True)
else:
    print('Skipping Hugging Face upload; set ENABLE_HF_UPLOAD=1 and HF_TOKEN to enable private adapter push.')
